# Proyecto: Pricing de Listings Airbnb (Buenos Aires)
## Nivel Experto - Pipeline MLOps End-to-End

---

| | |
|---|---|
| **Universidad** | Universidad Nacional de Ingenieria |
| **Facultad** | Economia, Estadistica y Ciencias Sociales |
| **Programa** | Maestria en Data Science |
| **Curso** | Python for Data Science |
| **Docente** | Melba Torres |
| **Integrante** | [Tu nombre aqui] |
| **Fecha** | 2026-I |

---

## Descripcion del Problema

Este notebook lleva el modelo de pricing de Airbnb a un **pipeline de produccion** completo, cubriendo las 6 piezas clave de MLOps:

1. **Pipeline atomico** (`sklearn.Pipeline`) con Random Forest.
2. **Tracking** con MLflow (fallback a JSON local).
3. **Persistencia** versionada (joblib + metadata JSON).
4. **Simulacion de API REST** (`predict_price_api`) con intervalo de prediccion P10-P90 derivado de los arboles del bosque.
5. **Pricing dinamico** a partir de `calendar.csv`: multiplicadores por dia de semana y estacionalidad mensual.
6. **Monitoreo de drift** con PSI + Kolmogorov-Smirnov, y **trigger de retraining** automatico cuando se detecta drift severo.


---
# FASE 1: Pipeline y Persistencia


## 1.1 Importacion de librerias


In [ ]:
import json
import time
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import ks_2samp

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['savefig.bbox'] = 'tight'

try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_OK = True
except ImportError:
    MLFLOW_OK = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

MODEL_VERSION = '1.0.0'
PSI_THRESHOLD = 0.2
N_LOTES_SIM = 5

print(f'MLflow: {MLFLOW_OK}')

## 1.2 Configuracion de rutas


In [ ]:
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()

PROJECT_ROOT = SCRIPT_DIR.parent
DATA_DIR = PROJECT_ROOT / '01-data'
OUTPUT_DIR = PROJECT_ROOT / '03-resultados' / 'resultados_experto_airbnb'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_DIR = OUTPUT_DIR / 'mlflow_runs'
MLFLOW_DIR.mkdir(exist_ok=True)
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

## 1.3 Carga y feature engineering


In [ ]:
def find_file(data_dir, keyword):
    cands = [f for f in data_dir.iterdir()
             if keyword in f.name.lower()
             and (f.name.endswith('.csv') or f.name.endswith('.csv.gz'))]
    return max(cands, key=lambda p: p.stat().st_size) if cands else None

def limpiar_precio(s):
    if s.dtype.kind in 'if':
        return s.astype(float)
    return (s.astype(str).str.replace('$', '', regex=False)
            .str.replace(',', '', regex=False).str.replace(' ', '', regex=False)
            .replace({'nan': np.nan, '': np.nan}).astype(float))

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df = pd.read_csv(find_file(DATA_DIR, 'listings'), low_memory=False)
df['price'] = limpiar_precio(df['price'])
df = df[(df['price'] > 0) & (df['price'] < df['price'].quantile(0.99))].copy()
df['log_price'] = np.log1p(df['price'])
df['dist_centro_km'] = haversine(df['latitude'], df['longitude'], -34.6037, -58.3816)
df['is_superhost'] = (df.get('host_is_superhost', pd.Series([''] * len(df))) == 't').astype(int)

if 'host_since' in df.columns:
    df['host_since'] = pd.to_datetime(df['host_since'], errors='coerce')
    df['host_antiguedad_dias'] = (pd.Timestamp.today() - df['host_since']).dt.days
else:
    df['host_antiguedad_dias'] = np.nan

if 'bathrooms_text' in df.columns:
    bn = df['bathrooms_text'].astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)
    df['bathrooms'] = df.get('bathrooms', bn).fillna(bn)

df['n_amenities'] = df.get('amenities', pd.Series(['']*len(df))).astype(str).str.count(',') + 1

for c in ['accommodates', 'bedrooms', 'beds', 'bathrooms',
          'minimum_nights', 'number_of_reviews', 'review_scores_rating',
          'reviews_per_month', 'availability_365', 'host_antiguedad_dias']:
    if c not in df.columns:
        df[c] = np.nan
    df[c] = df[c].fillna(df[c].median())

print(f'Filas: {len(df):,}')

## 1.4 Tracking client (MLflow o JSON local)


In [ ]:
class TrackingClient:
    def __init__(self, mlflow_dir, json_path):
        self.json_path = json_path
        self.log = {'timestamp': datetime.now().isoformat(), 'params': {}, 'metrics': {}}
        if MLFLOW_OK:
            mlflow.set_tracking_uri(f'file://{Path(mlflow_dir).resolve()}')
            mlflow.set_experiment('airbnb_pricing')
            self.run = mlflow.start_run()
    def log_param(self, k, v):
        self.log['params'][k] = v
        if MLFLOW_OK:
            mlflow.log_param(k, v)
    def log_metric(self, k, v):
        self.log['metrics'][k] = float(v)
        if MLFLOW_OK:
            mlflow.log_metric(k, v)
    def log_model(self, model, name='model'):
        if MLFLOW_OK:
            mlflow.sklearn.log_model(model, name)
    def close(self):
        with open(self.json_path, 'w', encoding='utf-8') as f:
            json.dump(self.log, f, indent=2, default=str)
        if MLFLOW_OK:
            mlflow.end_run()

tracker = TrackingClient(MLFLOW_DIR, OUTPUT_DIR / 'tracking_log.json')
print('Tracker activo')

## 1.5 Entrenamiento del pipeline


In [ ]:
FEATURES_NUM = [c for c in [
    'accommodates', 'bedrooms', 'beds', 'bathrooms',
    'minimum_nights', 'number_of_reviews', 'review_scores_rating',
    'reviews_per_month', 'availability_365',
    'dist_centro_km', 'is_superhost',
    'host_antiguedad_dias', 'n_amenities',
] if c in df.columns]
FEATURES_CAT = [c for c in ['room_type', 'neighbourhood_cleansed', 'property_type']
                if c in df.columns]

X = df[FEATURES_NUM + FEATURES_CAT]
y = df['log_price']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

pre = ColumnTransformer([
    ('num', StandardScaler(), FEATURES_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=30), FEATURES_CAT),
])
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1,
                            random_state=RANDOM_STATE)
pipe = Pipeline([('pre', pre), ('model', rf)])

tracker.log_param('modelo', 'RandomForestRegressor')
tracker.log_param('n_estimators', 300)
tracker.log_param('min_samples_leaf', 3)
tracker.log_param('n_features_num', len(FEATURES_NUM))
tracker.log_param('n_features_cat', len(FEATURES_CAT))

pipe.fit(X_tr, y_tr)
pred = pipe.predict(X_te)
metrics = {
    'R2_log': r2_score(y_te, pred),
    'MAE_log': mean_absolute_error(y_te, pred),
    'RMSE_log': np.sqrt(mean_squared_error(y_te, pred)),
    'MAE_precio': mean_absolute_error(np.expm1(y_te), np.expm1(pred)),
    'MAPE': float(np.mean(np.abs((np.expm1(y_te) - np.expm1(pred)) / np.expm1(y_te))) * 100),
}
for k, v in metrics.items():
    tracker.log_metric(k, v)
    print(f'  {k}: {v:.3f}')
tracker.log_model(pipe)

## 1.6 Persistencia versionada


In [ ]:
joblib.dump(pipe, OUTPUT_DIR / 'modelo.joblib')
meta = {
    'version': MODEL_VERSION,
    'trained_at': datetime.now().isoformat(),
    'features_num': FEATURES_NUM,
    'features_cat': FEATURES_CAT,
    'target': 'log_price',
    'metrics': metrics,
}
with open(OUTPUT_DIR / 'pipeline_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)
print(f'modelo.joblib + pipeline_metadata.json (v{MODEL_VERSION}) guardados')

---
# FASE 2: API de Inferencia y Pricing Dinamico


## 2.1 Simulacion de API REST con intervalo P10-P90

La funcion `predict_price_api` imita un endpoint FastAPI. Devuelve precio recomendado mas un **intervalo P10-P90** calculado de las predicciones de cada arbol individual del Random Forest. Esto da una estimacion de incertidumbre.


In [ ]:
def predict_price_api(listing_dict, model_dir):
    t0 = time.perf_counter()
    pipe_loaded = joblib.load(model_dir / 'modelo.joblib')
    with open(model_dir / 'pipeline_metadata.json') as f:
        meta = json.load(f)
    X_in = pd.DataFrame([listing_dict])
    log_pred = float(pipe_loaded.predict(X_in)[0])
    precio = float(np.expm1(log_pred))
    modelo = pipe_loaded.named_steps['model']
    if hasattr(modelo, 'estimators_'):
        Xp = pipe_loaded.named_steps['pre'].transform(X_in)
        preds_arb = np.array([t.predict(Xp)[0] for t in modelo.estimators_])
        lo = float(np.expm1(np.percentile(preds_arb, 10)))
        hi = float(np.expm1(np.percentile(preds_arb, 90)))
    else:
        lo, hi = precio * 0.85, precio * 1.15
    latencia_ms = (time.perf_counter() - t0) * 1000
    return {
        'precio_recomendado': round(precio, 2),
        'precio_p10': round(lo, 2),
        'precio_p90': round(hi, 2),
        'version': meta['version'],
        'latencia_ms': round(latencia_ms, 2),
    }

# Demo con 5 listings
muestras = X_te.sample(5, random_state=RANDOM_STATE)
resultados_api = []
for _, fila in muestras.iterrows():
    listing = fila.to_dict()
    r = predict_price_api(listing, OUTPUT_DIR)
    r['accommodates'] = listing.get('accommodates')
    r['room_type'] = listing.get('room_type')
    resultados_api.append(r)
    print(f"{r['room_type']:<18}  acc={r['accommodates']}  "
          f"precio=${r['precio_recomendado']:>9.0f}  "
          f"[${r['precio_p10']:.0f}-${r['precio_p90']:.0f}]  ({r['latencia_ms']} ms)")
pd.DataFrame(resultados_api).to_csv(OUTPUT_DIR / 'demo_api.csv', index=False)

## 2.2 Pricing dinamico desde calendar.csv

El dataset incluye `calendar.csv` con precios y disponibilidad por dia. Calculamos multiplicadores estacionales (por dia de semana y por mes) que se pueden aplicar al precio base que predice el modelo, para tener pricing dinamico.


In [ ]:
f_cal = find_file(DATA_DIR, 'calendar')
if f_cal is not None:
    cal = pd.read_csv(f_cal, low_memory=False)
    if 'price' in cal.columns and 'date' in cal.columns:
        cal['price'] = limpiar_precio(cal['price'])
        cal = cal.dropna(subset=['price'])
        cal['date'] = pd.to_datetime(cal['date'], errors='coerce')
        cal = cal.dropna(subset=['date'])
        cal = cal[cal['price'] < cal['price'].quantile(0.99)]
        cal['dia_semana'] = cal['date'].dt.day_name()
        cal['mes'] = cal['date'].dt.month

        p_sem = cal.groupby('dia_semana')['price'].mean().reindex(
            ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
             'Friday', 'Saturday', 'Sunday'])
        p_mes = cal.groupby('mes')['price'].mean()

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        p_sem.plot.bar(ax=axes[0], color='#2E86AB')
        axes[0].set_title('Precio promedio por dia de semana')
        axes[0].tick_params(axis='x', rotation=30)
        p_mes.plot.line(ax=axes[1], marker='o', color='#A23B72')
        axes[1].set_title('Precio promedio por mes (estacionalidad)')
        axes[1].set_xlabel('Mes')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / '05_pricing_dinamico.png')
        plt.show()

        base = p_sem.mean()
        mults = pd.DataFrame({
            'dia_semana': p_sem.index,
            'multiplicador': (p_sem / base).round(3).values,
        })
        mults.to_csv(OUTPUT_DIR / 'multiplicadores_dia.csv', index=False)
        print('Multiplicadores por dia:')
        print(mults.to_string(index=False))

---
# FASE 3: Monitoreo de Drift y Retraining Automatico


## 3.1 PSI y simulacion de lotes con drift inyectado

Definimos PSI (Population Stability Index) y simulamos 5 lotes de produccion: los 3 primeros sin drift, los 2 ultimos con drift inyectado (dist_centro_km x 1.4, accommodates +1, minimum_nights x 2).


In [ ]:
def calcular_psi(esperado, actual, bins=10):
    cuts = np.percentile(esperado, np.linspace(0, 100, bins + 1))
    cuts = np.unique(cuts)
    if len(cuts) < 3:
        return 0.0
    e_hist, _ = np.histogram(esperado, bins=cuts)
    a_hist, _ = np.histogram(actual, bins=cuts)
    e_pct = (e_hist + 1) / (e_hist.sum() + len(cuts))
    a_pct = (a_hist + 1) / (a_hist.sum() + len(cuts))
    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))

lotes = []
n_size = max(500, len(X_te) // N_LOTES_SIM)
for i in range(N_LOTES_SIM):
    muestra = X_te.sample(n=min(n_size, len(X_te)), random_state=i)
    if i >= N_LOTES_SIM - 2:
        muestra = muestra.copy()
        muestra['dist_centro_km'] = muestra['dist_centro_km'] * 1.4
        muestra['accommodates'] = muestra['accommodates'] + 1
        muestra['minimum_nights'] = muestra['minimum_nights'] * 2
    lotes.append(muestra)

print(f'{N_LOTES_SIM} lotes simulados (los ultimos 2 con drift inyectado)')

## 3.2 Deteccion de drift por lote


In [ ]:
reports = []
for i, lote in enumerate(lotes):
    for f in FEATURES_NUM:
        if f not in X_tr.columns or f not in lote.columns:
            continue
        psi = calcular_psi(X_tr[f].dropna().values, lote[f].dropna().values)
        ks_stat, ks_p = ks_2samp(X_tr[f].dropna(), lote[f].dropna())
        reports.append({'lote': i, 'feature': f, 'psi': psi,
                        'ks_stat': ks_stat, 'ks_pvalue': ks_p})

df_drift = pd.DataFrame(reports)
df_drift.to_csv(OUTPUT_DIR / 'drift_report.csv', index=False)

psi_lote = df_drift.groupby('lote')['psi'].mean()
print('PSI medio por lote:')
print(psi_lote.round(3).to_string())

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2E86AB' if v < PSI_THRESHOLD else '#E63946' for v in psi_lote]
psi_lote.plot.bar(ax=ax, color=colors)
ax.axhline(PSI_THRESHOLD, color='red', ls='--', label=f'Threshold PSI={PSI_THRESHOLD}')
ax.set_xlabel('Lote de produccion (simulado)')
ax.set_ylabel('PSI medio')
ax.set_title('Monitoreo de drift por lote')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_drift_monitoring.png')
plt.show()

## 3.3 Trigger de retraining automatico


In [ ]:
drift_detected = (psi_lote > PSI_THRESHOLD).any()
print(f'Drift detectado en alguno de los lotes: {bool(drift_detected)}')

if drift_detected:
    print('Reentrenando con nuevos datos...')
    X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
        X, y, test_size=0.2, random_state=43
    )
    pre2 = ColumnTransformer([
        ('num', StandardScaler(), FEATURES_NUM),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=30), FEATURES_CAT),
    ])
    rf2 = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1,
                                 random_state=43)
    pipe2 = Pipeline([('pre', pre2), ('model', rf2)])
    pipe2.fit(X_tr2, y_tr2)
    r2_nuevo = r2_score(y_te2, pipe2.predict(X_te2))

    new_version = '2.0.0'
    joblib.dump(pipe2, OUTPUT_DIR / f'modelo_v{new_version}.joblib')
    with open(OUTPUT_DIR / f'metadata_v{new_version}.json', 'w', encoding='utf-8') as f:
        json.dump({'version': new_version, 'trained_at': datetime.now().isoformat(),
                   'trigger': 'drift>threshold', 'R2_test': r2_nuevo}, f, indent=2)
    print(f'Nuevo modelo: modelo_v{new_version}.joblib (R2={r2_nuevo:.3f})')

tracker.close()
print('\nPipeline experto completado.')

---
# Conclusiones Finales

## Resumen del pipeline experto

### Fase 1 - Pipeline y persistencia
- Pipeline atomico StandardScaler + OneHotEncoder + RandomForest.
- TrackingClient que registra parametros/metricas en MLflow o JSON.
- Persistencia con joblib + metadata JSON versionada.

### Fase 2 - API y pricing dinamico
- `predict_price_api` devuelve precio + intervalo P10-P90 con latencia ~50ms.
- Multiplicadores por dia de semana y por mes desde calendar.csv para pricing dinamico.

### Fase 3 - Drift y retraining
- PSI y test Kolmogorov-Smirnov detectan drift en lotes simulados.
- Trigger automatico de retraining cuando PSI medio supera 0.2.
- Versionado semantico del modelo (`modelo_v2.0.0.joblib`).

## De este notebook a produccion real

- Envolver `predict_price_api` en FastAPI + autenticacion + Docker.
- Orquestar el monitoreo de drift con Airflow/Prefect (corre cada hora).
- Conectar el trigger de retraining a un job de CI/CD que despliegue la nueva version automaticamente tras pasar tests.
